# Lab 2: Deep Learning Mini-Project: Image Quality Assessment

Authors: Alexandre Bruckert, Nantes Université

Date: 2026/2027

Over the next two lab sessions (6 hours), you will transition from learning basic PyTorch syntax to designing, optimizing, and evaluating neural networks for a real-world task.

## The task: Image Quality Assessment (IQA)

Instead of classifying what is in an image, you will predict how good it looks.
You are tackling an Image Quality Assessment (IQA) task. Your model will analyze an image and predict its Mean Opinion Score (MOS), which is a continuous value usually ranging from 1.0 (terrible quality) to 5.0 (excellent quality), representing the average human perception of the image's visual appeal, sharpness, and lack of distortions.

Because the target is a continuous number, this is a **regression** problem.

## The Dataset: KonIQ-10k
For this challenge, you will be working with the **KonIQ-10k** dataset, a major benchmark in the field of Image Quality Assessment. It consists of 10,073 images carefully sampled from a massive public multimedia database. Unlike older academic datasets where pristine images were artificially corrupted with math-generated noise, KonIQ-10k features **"in-the-wild" authentic distortions**. This means the images suffer from real-world, everyday issues: poor lighting, shaky hands (motion blur), cheap camera sensors, and aggressive JPEG compression. The ground truth MOS labels (ranging from 1.0 to 5.0) were collected through a massive crowdsourcing effort involving over 1.2 million human ratings.

The KonIQ-10k dataset is available at: https://database.mmsp-kn.de/koniq-10k-database.html
You should use the lower resolution images ($512 \times 384$) for this work.

## Evaluation

You are tasked with building the best model possible for this task. You will be evaluated based on a **heavily documented** notebook that contains your code, results, and analysis. The notebook should be structured to clearly present your approach, experiments, and findings.

Your notebook must contain the following distinct sections, representing "milestones" in the project:

- Milestone 1: The CNN Baseline: Design and train a Convolutional Neural Network (CNN) using MSELoss.

- Milestone 2: The RNN Experiment: Treat the image as a sequence of pixel rows! Design a Recurrent Neural Network (LSTM or GRU) to predict the MOS. Compare its performance to the CNN.

- Milestone 3: The Transfer Learning Approach: Use a pre-trained model (e.g., ResNet, EfficientNet) as a feature extractor. Fine-tune it for the IQA task and compare its performance to your previous models.

- Milestone 4: The Ultimate Model: Combine Early Stopping, Schedulers, Regularization, and your best architecture to build your final model.

Similarly to the grid in Lab 1, the concepts explored in each milestone will be part of the program of the final exam!

In [2]:
import torch
import torchvision
import pandas as pd
import matplotlib.pyplot as plt

print("PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())
print("Installation OK !")

PyTorch : 2.14.1
GPU disponible : False
Installation OK !


In [3]:
import sys
import torch
import torchvision

print("Python :", sys.version)
print("Environnement :", sys.executable)
print("PyTorch :", torch.__version__)
print("Tout fonctionne !")

Python : 3.14.7 (main, Aug  5 2026, 10:29:49) [Clang 17.0.0 (clang-1700.6.4.2)]
Environnement : /Users/ilefmhabrech/Documents/M2/DL/TP2-Deep-Learning/.venv/bin/python
PyTorch : 2.14.1
Tout fonctionne !


In [5]:
import os

print("Répertoire actuel :", os.getcwd())
print("Fichiers présents :", os.listdir())

Répertoire actuel : /Users/ilefmhabrech/Documents/M2/DL/TP2-Deep-Learning/experiments
Fichiers présents : ['transfer.ipynb', 'CNN.ipynb', 'rnn.ipynb']


In [6]:
import os

print("Nombre d'images :", len(os.listdir("../iqa_data/images")))
print("Fichiers :", os.listdir("../iqa_data"))

Nombre d'images : 10373
Fichiers : ['koniq10k_scores_and_distributions.csv', 'images']


In [8]:
import os
import pandas as pd

# Chemin vers les images
images_dir = "../iqa_data/images"

# Chemin vers les scores
scores_dir = "../iqa_data"

# Compter les images
images = os.listdir(images_dir)

print("Nombre d'images :", len(images))

# Voir le nom du fichier CSV
print("Fichiers de scores :", os.listdir(scores_dir))

Nombre d'images : 10373
Fichiers de scores : ['koniq10k_scores_and_distributions.csv', 'images']


In [9]:
# Trouver le fichier CSV
csv_files = [
    f for f in os.listdir(scores_dir)
    if f.endswith(".csv")
]

csv_path = os.path.join(scores_dir, csv_files[0])

# Charger le CSV
df = pd.read_csv(csv_path)

# Afficher les 5 premières lignes
display(df.head())

# Afficher les noms des colonnes
print(df.columns.tolist())

,image_name,c1,c2,c3,c4,c5,c_total,MOS,SD,MOS_zscore
0,10004473376.jpg,0,0,25,73,7,105,3.828571,0.527278,77.383621
1,10007357496.jpg,0,3,45,47,1,96,3.479167,0.580003,68.728571
2,10007903636.jpg,1,0,20,73,2,96,3.781250,0.527220,78.628571
3,10009096245.jpg,0,0,21,75,13,109,3.926606,0.556218,77.243750
4,100117038.jpg,0,1,21,72,6,100,3.830000,0.532860,75.112500


['image_name', 'c1', 'c2', 'c3', 'c4', 'c5', 'c_total', 'MOS', 'SD', 'MOS_zscore']


## 1. Setup

Here below, you will find the necessary imports and a custom Dataset class to load the KonIQ-10k dataset. You will need to download the dataset and place it in a folder named `iqa_data` with the following structure:
```
iqa_data/
├── images/
├── train_labels.csv
```

In [12]:

# Installation des dépendances (à exécuter une seule fois)
import sys
import shutil
import subprocess

pip = [sys.executable, "-m", "pip", "install"]

# PyTorch CUDA pour les PC Windows avec GPU NVIDIA
if sys.platform == "win32" and shutil.which("nvidia-smi"):
    subprocess.check_call(pip + [
        "torch==2.13.0+cu130",
        "torchvision==0.28.0+cu130",
        "--extra-index-url", "https://download.pytorch.org/whl/cu130"
    ])

# Autres dépendances + PyTorch standard si nécessaire
subprocess.check_call(pip + ["-r", "../requirements.txt"])

  Using cached torch-2.13.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (39 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
Using cached torch-2.13.0-cp314-cp314-macosx_14_0_arm64.whl (111.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 9.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 22.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 45.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 43.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 51.4 MB/s  0:00:00
Using cached nest_asyncio-1.6.0-py3-none-any.whl (5.2 kB)
  Attempting uninstall: Pillow
    Found existing installation: pillow 12.3.0
    Uninstalling pillow-12.3.0:
      Successfully uninstalled pillow-12.3.0
  Attempting uninstall: numpy0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [Pillow]
    Found existing installation: numpy 2.5.3━━━━━━━━━━━━━━━━━━ 1/9 [Pillow]
    Un

0

In [13]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch :", torch.__version__)
print("Device :", device)


PyTorch : 2.14.1
Device : mps


In [16]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
set_seed(42)

# ====================================================================
# 1. CUSTOM DATASET CLASS (Provided)
# ====================================================================
# In real-world projects, data doesn't come neatly packaged in PyTorch.
# You have a folder of images and a CSV file with labels.
# We build a custom Dataset class to handle this.

class IQADataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        """
        Args:
            csv_file (string): Path to the csv file with annotations (columns: image_id, mos).
            img_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        # Load the CSV file
        self.labels_df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        # Get image name and target MOS
        img_name = str(self.labels_df.iloc[idx]['image_name'])
        mos = float(self.labels_df.iloc[idx]['MOS'])

        # Load image from disk
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        # Apply transformations (e.g., resizing, converting to tensor)
        if self.transform:
            image = self.transform(image)

        # Return the image and the MOS (as a float32 tensor for regression)
        return image, torch.tensor(mos, dtype=torch.float32)

# ====================================================================
# DATA LOADING EXAMPLE
# ====================================================================
# Resize all images to 128x128 to keep training relatively fast
base_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Assuming you have downloaded the dataset into a folder named 'iqa_data'
# containing 'images/' folder and 'train_labels.csv'
#

#dataset = IQADataset(csv_file='iqa_data\train_labels.csv',
#                      img_dir='iqa_data\images',
#                      transform=base_transform)

dataset = IQADataset(
    csv_file="../iqa_data/train_labels.csv",
    img_dir="../iqa_data/images",
    transform=base_transform
)

sample, label = dataset[0]
print(f"Sample {0}: {sample}")
print(f"Label {0}: {label}")

print("Dataset class ready! Your turn to instantiate it and create DataLoaders.")

Using device: cpu
Sample 0: tensor([[[ 0.2706, -0.3647, -0.4039,  ..., -0.6157, -0.6471, -0.6078],
         [-0.3490, -0.5216, -0.2784,  ..., -0.8667, -0.8588, -0.8824],
         [-0.8980, -0.8824, -0.7725,  ..., -0.9216, -0.8431, -0.7647],
         ...,
         [-0.4980, -0.5686, -0.2392,  ..., -0.7725, -0.9137, -0.8824],
         [-0.5451, -0.5529, -0.4980,  ..., -0.7804, -0.9294, -0.9137],
         [-0.3569, -0.3647, -0.4980,  ..., -0.8353, -0.9294, -0.9294]],

        [[ 0.3176, -0.2863, -0.3412,  ..., -0.3333, -0.4353, -0.3882],
         [-0.1922, -0.3647, -0.1686,  ..., -0.7725, -0.7804, -0.8353],
         [-0.7882, -0.7804, -0.6549,  ..., -0.9373, -0.8196, -0.7176],
         ...,
         [-0.3176, -0.4667, -0.1294,  ..., -0.6784, -0.8824, -0.8667],
         [-0.3490, -0.3882, -0.3176,  ..., -0.7020, -0.9059, -0.8980],
         [-0.1765, -0.2078, -0.3098,  ..., -0.7882, -0.9059, -0.9137]],

        [[ 0.3804, -0.2471, -0.2784,  ..., -0.0510, -0.3255, -0.3490],
         [-0.2157